In [1]:
import numpy as np
import pandas as pd

In [2]:
path = r"C:\Users\JOKER\Desktop\berlin\Fächer\Summer School\Power Markets\Data"
da = pd.read_csv(f"{path}/day_ahead_prices.csv")
load = pd.read_csv(f"{path}/forecast_load.csv")

In [3]:
grid = da[["timestamp_utc", "timestamp_local", "day_ahead_price_eur_mwh"]].merge(
    load[["timestamp_utc", "total_forecast_mwh"]], on="timestamp_utc")

In [4]:
grid["local_date"] = grid["timestamp_local"].str[:10]
event_date = grid.groupby("local_date")["day_ahead_price_eur_mwh"].mean().idxmax()
event = grid[grid["local_date"] == event_date].reset_index(drop=True)
print("event day:", event_date)
print("mean price:", round(event["day_ahead_price_eur_mwh"].mean(), 2), "EUR/MWh")
print("max quarter-hour price:", round(event["day_ahead_price_eur_mwh"].max(), 2), "EUR/MWh")
print("year mean price:", round(grid["day_ahead_price_eur_mwh"].mean(), 2), "EUR/MWh")
print("forecast load that day:", round(event["total_forecast_mwh"].sum(), 2), "MWh")

event day: 2024-12-12
mean price: 395.34 EUR/MWh
max quarter-hour price: 936.28 EUR/MWh
year mean price: 78.51 EUR/MWh
forecast load that day: 132.44 MWh


In [5]:
day_mask = (grid["local_date"] == event_date).to_numpy()
P = event["day_ahead_price_eur_mwh"].to_numpy()
out = event[["timestamp_local", "day_ahead_price_eur_mwh", "total_forecast_mwh"]].copy()

In [6]:
for name, tag in [("UNHEDGED", "None"), ("COARSE", "coarse"), ("GRANULAR", "granular")]:
    if tag == "None":
        H = np.zeros(day_mask.sum())
    else:
        H = pd.read_csv(f"{path}/task3_hedge_" + tag + ".csv")["hedge_energy"].to_numpy()[day_mask]
    R = pd.read_csv(f"{path}/residual_" + tag + ".csv")["day_ahead_residual"].to_numpy()[day_mask]

    # Cost of buying the leftover volume at that day's prices
    print("\n", name)
    print("  hedge energy:", round(H.sum(), 2), "MWh")
    print("  net residual:", round(R.sum(), 2), "MWh")
    print("  absolute residual:", round(np.abs(R).sum(), 2), "MWh")
    print("  residual Day-Ahead cost:", round(R @ P, 2), "EUR")
    out["hedge_" + tag] = H
    out["residual_" + tag] = R


 UNHEDGED
  hedge energy: 0.0 MWh
  net residual: 132.44 MWh
  absolute residual: 132.44 MWh
  residual Day-Ahead cost: 61435.8 EUR

 COARSE
  hedge energy: 127.43 MWh
  net residual: 5.01 MWh
  absolute residual: 32.87 MWh
  residual Day-Ahead cost: 6305.71 EUR

 GRANULAR
  hedge energy: 136.19 MWh
  net residual: -3.75 MWh
  absolute residual: 31.33 MWh
  residual Day-Ahead cost: 2259.99 EUR


In [7]:
out.to_csv(f"{path}/event_day.csv", index=False)